# Section Labeling Pipeline
Classifies paper sections using the OpenAI Batch API.

**Steps:**
1. Configure paths and parameters
2. Prepare batch JSONL files (splits by token limit)
3. Upload files and submit batch jobs
4. Poll batch status until completion
5. Download and save results

## 1. Setup & Configuration

In [24]:
import os
import json
import time
import pandas as pd
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

# --- DATA ---
FILE_PATH        = "auto_eval_sample.parquet"
PAPER_ID_COL     = 'corpusid'
HEADER_COL       = 'extracted'
CONTENT_COL      = 'section_text'

# --- MODEL ---
MODEL_NAME       = "gpt-5-mini"
TOKEN_LIMIT      = 300   # max tokens per content snippet

# --- BATCHING ---
# Stay below the 2M enqueued-token hard limit
BATCH_TOKEN_LIMIT   = 1_800_000
BATCH_OUTPUT_DIR    = "batch_input"
BATCH_OUTPUT_BASE   = os.path.join(BATCH_OUTPUT_DIR, "batch_input")   # → batch_input_1.jsonl, etc.
RESULTS_OUTPUT_DIR  = "output"

os.makedirs(RESULTS_OUTPUT_DIR, exist_ok=True)

In [25]:
# --- PROMPT ---
PROMPT_TEMPLATE = """
# TASK
Analyze the CURRENT_CONTENT below. Use the content snippet and the context from the previous and following headers to assign the most accurate section label. The snippet contains the first 300 words of the text.

# SECTION DEFINITIONS
- `introduction`: Introduces the topic, problem, research questions, motivation, and/or paper structure.
- `lit_review`: Discusses previous work and the state-of-the-art. It is also the main body of papers that only review existing literature without presenting new findings.
- `methods`: How the research was done (e.g., experiments, data, algorithms).
- `results`: Presents findings and data without interpretation.
- `discussion`: Interprets the results, discusses implications, and limitations.
- `conclusion`: Summarizes the paper and often discusses future work.
- `case_report`: In Medicine papers, a narrative of a patient case or group of related cases. Often includes patient profiles, unique events, or specific interventions/treatments and their outcomes.
- `development`: The core argumentative or theoretical advancement of a paper that isn't presenting empirical methods or results. Common in fields like philosophy, theoretical math, or law. Only use it if the other labels really don't apply.
- `something_else`: Other administrative sections that do not fit in any of the other labels. Avoid using it whenever possible.
- `ambiguous`: Use this label only as a *last resort* if it's impossible to determine the section's purpose from the given text and context.

# IMPORTANT NOTES
- A single logical section (like 'Methods') might be broken into several smaller headers (e.g., '3.1 Study Design', then '3.2 Participants'). Your job is to label *each* of these individual headers with the parent label, which would be `methods` in this case. 
- Not all papers have all sections! Most will only have a subset. A computer science paper might not have a separate `lit_review`, and a philosophy paper won't have experimental `results`. Your task is to label what's there, not what "should" be there.
- Always choose a category from the SECTION DEFINITIONS, never invent a new label.

# RESPONSE
Provide your response in a JSON format with these keys:
1. "label": Your chosen section label (string).
2. "confidence": Your confidence in this classification (string: "high", "medium", or "low").

---

# EXAMPLE
## CONTEXT
  - PREVIOUS_HEADERS:
    "- 1. Introduction
     - 2. Related Work
     - 2.1. Neural Network Pruning
     - 2.2. Quantization Aware Training
     - 3. Our Proposed Method"

==> CURRENT ROW IS HERE
## CURRENT_CONTENT
    "3.1. Dataset
We use the publicly available ImageNet (ILSVRC 2012) dataset, which consists of over 1.2 million training images from 1,000 classes. The images vary in resolution and are resized to 224x224 pixels during preprocessing."

  - FOLLOWING_HEADERS:
"      - 3.2. Model Architecture
     - 3.3. Training Procedure
     - 4. Experiments and Results
     - 4.1. Main Findings
     - 4.2. Ablation Studies"

**Response:**
{{
  "label": "methods",
  "confidence": "high"
}}

---
# YOUR TURN
## CONTEXT
  - PREVIOUS_HEADERS:
{previous_headers_list}

==> CURRENT ROW IS HERE
- CURRENT_CONTENT:
{section_content_snippet}

  - FOLLOWING_HEADERS:
{following_headers_list}

*Response:*
"""

In [26]:
# --- MODEL PARAMETERS (static per request) ---
MODEL_PARAMETERS = {
    "text": {
        "format": {
            "type": "json_schema",
            "name": "labeled_confidence_response",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "label": {"type": "string", "description": "Chosen section label"},
                    "confidence": {
                        "type": "string",
                        "description": "Confidence in the classification (high, medium, or low)",
                        "enum": ["high", "medium", "low"]
                    }
                },
                "required": ["label", "confidence"],
                "additionalProperties": False
            }
        },
        "verbosity": "medium"
    },
    "temperature": 0,
    "reasoning": {"effort": "low"},
    "tools": [],
    "store": True
}

print("✅ Configuration done.")

✅ Configuration done.


## 2. Prepare Batch Files

The Batch API is ~50% cheaper than real-time requests and has higher rate limits. The tradeoff is async execution with up to 24h completion window.

OpenAI enforces a 2M enqueued-token limit across all in-progress batches in the org. We split requests into multiple files so each file stays safely under that ceiling. Files are submitted one at a time (or you can wait for one to finish before submitting the next).

In [27]:
try:
    encoding = tiktoken.get_encoding("o200k_base")
except Exception as e:
    print(f"Error getting tokenizer: {e}")
    exit()

# Load and filter data
df = pd.read_parquet(FILE_PATH)
print(f"✅ Loaded {len(df)} rows.")

# Remove irrelevant sections - we don't need the LLM to label those, it would just waste tokens and budget
df = df[
    (df[CONTENT_COL].str.strip() != "") &
    (~df['sec_label_extended'].isin(['figure_table', 'ending', 'other']))
]
print(f"   After filtering: {len(df)} rows.")

✅ Loaded 27028 rows.
   After filtering: 15941 rows.


In [28]:
def write_batch_file(requests_list: list, file_counter: int, base_name: str) -> str:
    """Write a list of request dicts to a numbered JSONL file. Returns the filename."""
    filename = f"{base_name}_{file_counter}.jsonl"
    with open(filename, "w") as f:
        for req in requests_list:
            f.write(json.dumps(req) + "\n")
    print(f"  ✅ {filename}  ({len(requests_list)} requests)")
    return filename


all_input_data    = []   # for mapping results back to originals
batch_filenames   = []   # list of written JSONL paths

file_counter          = 1
current_batch_reqs    = []
current_batch_tokens  = 0
total_requests        = 0

In [29]:
print("\nPreparing batch files...")
for paper_id, group in df.groupby(PAPER_ID_COL):
    group = group.reset_index(drop=True)

    for index, row in group.iterrows():
        # Build header context windows
        prev_headers = group.iloc[max(0, index - 5):index][HEADER_COL].tolist()
        next_headers = group.iloc[index + 1:min(len(group), index + 6)][HEADER_COL].tolist()
        prev_str = "\n".join(f" - {h} " for h in prev_headers)
        next_str = "\n".join(f" - {h} " for h in next_headers)

        # Truncate content snippet
        full_content = str(row[CONTENT_COL])
        tokens = encoding.encode(full_content)
        snippet = encoding.decode(tokens[:TOKEN_LIMIT]) if len(tokens) > TOKEN_LIMIT else full_content

        # Format prompt
        prompt = PROMPT_TEMPLATE.format(
            previous_headers_list=prev_str,
            section_content_snippet=snippet,
            following_headers_list=next_str
        )
        prompt_tokens = len(encoding.encode(prompt))

        # Split into a new file if we'd exceed the token limit
        if current_batch_reqs and (current_batch_tokens + prompt_tokens > BATCH_TOKEN_LIMIT):
            batch_filenames.append(
                write_batch_file(current_batch_reqs, file_counter, BATCH_OUTPUT_BASE)
            )
            file_counter += 1
            current_batch_reqs   = []
            current_batch_tokens = 0

        custom_id = f"{paper_id}-{index}"

        request_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL_NAME,
                "input": [{
                    "role": "user",
                    "content": [{"type": "input_text", "text": prompt}]
                }],
                **MODEL_PARAMETERS
            }
        }

        current_batch_reqs.append(request_data)
        current_batch_tokens += prompt_tokens
        total_requests += 1
        all_input_data.append({"custom_id": custom_id, "original_header": row[HEADER_COL]})

# Write last batch
if current_batch_reqs:
    batch_filenames.append(
        write_batch_file(current_batch_reqs, file_counter, BATCH_OUTPUT_BASE)
    )

input_df = pd.DataFrame(all_input_data)
print(f"\n✅ Done. {total_requests} total requests across {len(batch_filenames)} batch file(s).")


Preparing batch files...
  ✅ batch_input\batch_input_1.jsonl  (1775 requests)
  ✅ batch_input\batch_input_2.jsonl  (1781 requests)
  ✅ batch_input\batch_input_3.jsonl  (1786 requests)
  ✅ batch_input\batch_input_4.jsonl  (1810 requests)
  ✅ batch_input\batch_input_5.jsonl  (1811 requests)
  ✅ batch_input\batch_input_6.jsonl  (1802 requests)
  ✅ batch_input\batch_input_7.jsonl  (1794 requests)
  ✅ batch_input\batch_input_8.jsonl  (1796 requests)
  ✅ batch_input\batch_input_9.jsonl  (1586 requests)

✅ Done. 15941 total requests across 9 batch file(s).


## 3. Upload Files & Submit Batch Jobs

In [30]:
submitted_batches = []   # list of (batch_id, filename)

for filename in batch_filenames:
    # Upload
    with open(filename, "rb") as f:
        file_obj = client.files.create(file=f, purpose="batch")

    # Submit
    batch = client.batches.create(
        input_file_id=file_obj.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"description": f"section-labeling | {os.path.basename(filename)}"}
    )

    submitted_batches.append((batch.id, filename))
    print(f"  Submitted  batch_id={batch.id}  file={os.path.basename(filename)}")

print(f"\n✅ Submitted {len(submitted_batches)} batch job(s).")

  Submitted  batch_id=batch_69dd78810eec8190b6897ac3e5516061  file=batch_input_1.jsonl
  Submitted  batch_id=batch_69dd7882419881909908195c51203eb6  file=batch_input_2.jsonl
  Submitted  batch_id=batch_69dd788391f48190b2c4c0b144df941e  file=batch_input_3.jsonl
  Submitted  batch_id=batch_69dd78850b948190a61bed9c71cc5317  file=batch_input_4.jsonl
  Submitted  batch_id=batch_69dd788708448190b06b6dd90283c47a  file=batch_input_5.jsonl
  Submitted  batch_id=batch_69dd78883fb88190b0eee9f7e992450d  file=batch_input_6.jsonl
  Submitted  batch_id=batch_69dd7889487c819084725bc44b649e82  file=batch_input_7.jsonl
  Submitted  batch_id=batch_69dd788ae47c8190bc44ae1e809bce1d  file=batch_input_8.jsonl
  Submitted  batch_id=batch_69dd788c3d6c8190ae0e3aea84154980  file=batch_input_9.jsonl

✅ Submitted 9 batch job(s).


## 4. Poll Until All Batches Complete

Submit one batch at a time to avoid hitting the enqueued-token org limit.

We wait for each batch to complete before submitting the next.

In [ ]:
done = []

for filename in batch_filenames:
    # Upload & submit
    with open(filename, "rb") as f:
        file_obj = client.files.create(file=f, purpose="batch")
    batch = client.batches.create(
        input_file_id=file_obj.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"description": f"section-labeling | {os.path.basename(filename)}"}
    )
    print(f"Submitted {batch.id} for {os.path.basename(filename)}")

    # Poll until this batch finishes before moving on
    while True:
        batch = client.batches.retrieve(batch.id)
        print(f"  status={batch.status}  {batch.request_counts.completed}/{batch.request_counts.total}")
        if batch.status in {"completed", "failed", "expired", "cancelled"}:
            break
        time.sleep(60)

    done.append((batch.id, filename, batch))

Submitted batch_69dd789be4b88190ba6254bd1beaba37 for batch_input_1.jsonl
  status=validating  0/0


## 5. Download Results

In [16]:
for i, (batch_id, filename, batch) in enumerate(done, start=1):
    if batch.status != "completed" or not batch.output_file_id:
        print(f"  ⚠️  Skipping batch {batch_id} (status={batch.status}, no output file)")
        continue

    file_response = client.files.content(batch.output_file_id)
    out_path = os.path.join(RESULTS_OUTPUT_DIR, f"{i}.txt")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(file_response.text)

    print(f"  ✅ Saved results for batch {batch_id} → {out_path}")

print("\n✅ All results downloaded.")

  ⚠️  Skipping batch batch_69dd6f3736408190b7f01fdba5e705bf (status=failed, no output file)
  ⚠️  Skipping batch batch_69dd6f3aa5708190965b795d338e7b3e (status=failed, no output file)
  ⚠️  Skipping batch batch_69dd6f3fc6788190a97948744ab30a14 (status=failed, no output file)
  ⚠️  Skipping batch batch_69dd6f41535c8190aeb5fe38e2fe8acc (status=failed, no output file)
  ⚠️  Skipping batch batch_69dd6f432df481909d7d98cb0b6757c9 (status=failed, no output file)
  ⚠️  Skipping batch batch_69dd6f45052c819096d81264dbdbf15f (status=failed, no output file)
  ✅ Saved results for batch batch_69dd6f3cee648190933fb6090ee858fe → output\7.txt
  ✅ Saved results for batch batch_69dd6f3e78b88190bbcd4ee3224feac9 → output\8.txt
  ✅ Saved results for batch batch_69dd6f38ff088190a2f4b6dd9615dd8f → output\9.txt

✅ All results downloaded.
